# Laboratorio 00. Introducción a Deep Learning

**Pregunta central:** ¿Qué aprende una red a partir de ejemplos y cómo comprobamos que no solo memorizó entrenamiento?

## Objetivos

- Distinguir datos, parámetros, predicción, pérdida y actualización.
- Comparar una red sencilla con un baseline.
- Separar entrenamiento, validación y prueba antes de ajustar.

## Prerrequisitos y conexión

Inicio del módulo. Basta Python básico y álgebra elemental; el módulo de Machine Learning aporta experiencia con variables, particiones y métricas.

## Teoría breve

Un modelo $f_\theta$ transforma una entrada en una predicción. El ajuste busca parámetros que reduzcan una pérdida promedio, y la evaluación en datos no usados estima generalización.

## Problema y datos

Se genera una curva sintética no lineal con ruido gaussiano fijo por semilla. Una permutación reproducible separa los ejemplos en entrenamiento, validación y prueba; no se descarga ningún recurso.

## Evaluación

Compara RMSE de la red con el predictor constante del promedio de train. Revisa curvas de pérdida y predicciones en test; no uses test para elegir épocas.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Generar datos y referencia
Separamos cronológicamente un conjunto sintético; el baseline usa solo el promedio de entrenamiento.


In [ ]:
rng = np.random.default_rng(SEED)
x = np.linspace(-2, 2, 360, dtype=np.float32).reshape(-1, 1)
y = (0.7 * x**2 - 0.4 * x + 0.3 * np.sin(3 * x) + rng.normal(0, 0.15, size=x.shape)).astype(np.float32)
order = rng.permutation(len(x))
x, y = x[order], y[order]
x = torch.from_numpy(x)
y = torch.from_numpy(y)
cut_train = int(0.70 * len(x))
cut_validation = int(0.85 * len(x))
x_train, x_validation, x_test = x[:cut_train], x[cut_train:cut_validation], x[cut_validation:]
y_train, y_validation, y_test = y[:cut_train], y[cut_train:cut_validation], y[cut_validation:]
baseline_value = y_train.mean()
print('tamaños train/validación/test:', len(x_train), len(x_validation), len(x_test))
print('RMSE baseline validación:', torch.mean((baseline_value - y_validation) ** 2).sqrt().item())
print('RMSE baseline test:', torch.mean((baseline_value - y_test) ** 2).sqrt().item())

**Resultado e interpretación.** La partición deja las últimas observaciones para evaluar generalización. El baseline fija una referencia que el modelo debe superar.


### Aprender y evaluar
Entrenamos una capa lineal con MSE y hacemos visible el ciclo de optimización.


In [ ]:
model = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1)).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_history, validation_history = [], []
for epoch in range(160):
    model.train()
    prediction = model(x_train.to(device))
    loss = loss_fn(prediction, y_train.to(device))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    model.eval()
    with torch.no_grad():
        validation_prediction = model(x_validation.to(device))
        validation_history.append(loss_fn(validation_prediction, y_validation.to(device)).item())
model.eval()
with torch.no_grad():
    validation_prediction = model(x_validation.to(device)).cpu()
    test_prediction = model(x_test.to(device)).cpu()
rmse_validation = torch.mean((validation_prediction - y_validation) ** 2).sqrt().item()
rmse_test = torch.mean((test_prediction - y_test) ** 2).sqrt().item()
print('RMSE modelo validación:', round(rmse_validation, 4))
print('RMSE modelo test:', round(rmse_test, 4))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(loss_history, label='train')
axes[0].plot(validation_history, label='validación')
axes[0].set(xlabel='Época', ylabel='MSE', title='Curvas de aprendizaje')
axes[0].legend()
axes[1].scatter(x_test.numpy(), y_test.numpy(), label='real')
axes[1].scatter(x_test.numpy(), test_prediction.numpy(), label='predicho')
axes[1].set(xlabel='x', ylabel='y', title='Predicciones en test')
axes[1].legend()
plt.tight_layout()
plt.show()

**Resultado e interpretación.** Compara el RMSE de prueba con el baseline y revisa si la pérdida de entrenamiento desciende. Una pérdida baja en train no basta por sí sola.


## Limitaciones y conclusiones

El ejemplo sintético permite recorrer el ciclo completo, pero no representa la complejidad de una aplicación real. La métrica de prueba estima el error solo para esta generación de datos y esta semilla. Compara el resultado ejecutado con el baseline; no presupongas que la red ganará en toda muestra.

## Ejercicios

1. **Guiado:** cambia el ruido de `0.15` a `0.35` y compara RMSE de validación y prueba con el baseline.
2. **Independiente:** sustituye la red por una sola capa lineal y explica la forma que puede representar.
3. **Desafío:** cambia la separación aleatoria por una partición por rangos de `x` y analiza qué tipo de generalización mide.

## Conexión con el siguiente laboratorio

El próximo paso inspecciona tensores, dispositivos y derivadas automáticas, que permiten expresar en PyTorch las operaciones de este ciclo.